In [ ]:
import time
import gc
import os
import torch
import pandas as pd
from huggingface_hub import login
from datasets import load_dataset
from transformers import BitsAndBytesConfig
from transformers import EarlyStoppingCallback
from peft import prepare_model_for_kbit_training
from langchain_community.vectorstores import FAISS
from transformers import TrainingArguments, Trainer
from langchain_core.documents.base import Document
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM, AutoConfig
from transformers import AutoTokenizer, AutoModelForCausalLM
from langchain_community.embeddings import HuggingFaceBgeEmbeddings



# Load the tokenizer and model
#model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"
model_name='microsoft/phi-4'
#model_name='meta-llama/Llama-3.1-8B-Instruct'

train = pd.read_excel("./ollamawork/Finetuning_2025/Finetune_colama-llama3.1/train_750_library_files.xlsx",sheet_name='train')
val = pd.read_excel("./ollamawork/Finetuning_2025/Finetune_colama-llama3.1/train_750_library_files.xlsx",sheet_name='eval')


train_dataset = train.iloc[:, :]
test_dataset = val.iloc[:, :]

col = {'Endpoint':'endpoint', 'Description': 'question', 'Test_Code': 'answer', 'Inputs': 'inputs', 'Output':'output'}
train_dataset.rename(columns = col, inplace=True)
test_dataset.rename(columns = col, inplace=True)

train_dataset.to_csv('train.csv', index = False)
test_dataset.to_csv('test.csv', index = False)


data_files = {"train": "train.csv", "test": "test.csv"}
dataset = load_dataset("csv", data_files=data_files)



# Setting up the tokenizer for Phi-2
tokenizer=AutoTokenizer.from_pretrained(
    model_name,
    add_eos_token=True,
    trust_remote_code=False
)

tokenizer.pad_token=tokenizer.eos_token
tokenizer.truncation_side="left"


def get_between_triple_quotes(string):
    triple_quote = '"""'
    start = string.find(triple_quote)  # Find the first triple quote
    end = string.find(triple_quote, start + 3)  # Find the next triple quote after the first
    if start != -1 and end != -1 and end > start:
        return string[start + 3:end].strip()  # Extract the content between the triple quotes
    else:
        return ""  # Return an empty string if triple quotes are not found

    
    
def filt_s(s):
    if s:  # Check if the string is not empty
        substrings = s.split('\n')  # Split the string by newlines
        if len(substrings) > 1:  # Check if there is more than one line
            desired_substring = substrings[1]  # Get the second line
            cleaned_substring = desired_substring.strip()  # Strip leading/trailing whitespace
            return cleaned_substring
        else:
            return s.strip()  # Return the string itself if it's a single line
    else:
        return '' 
            


def collate_and_tokenize(examples):
    batch_encoded = []
    labels = []
    for i in range(len(examples['endpoint'])):
        endpoint = examples['endpoint'][i]
        question = examples['question'][i]
        inputs = examples['inputs'][i]
        output = examples['output'][i]
        answer = examples['answer'][i]

        # Apply string processing functions
        desc = filt_s(get_between_triple_quotes(answer))

        # Format the prompt
        prompt = f"""
<|im_start|>system<|im_sep|> 
Write down a python function that tests the API scenario described below. Make sure you use appropriate docstring while writing code. Below are the endpoint, inputs, and output of the function as well.<|im_end|>
<|im_start|>user<|im_sep|>
Endpoint:
{endpoint}\n
Description:
{question}\n
{desc}\n
Inputs:
{inputs}\n
Output:
{output}\n
Answer:
{answer}<|im_end|>
<|im_start|>assistant<|im_sep|>"""
        
        # Tokenize the prompt
        encoded = tokenizer(
            prompt,
            return_tensors="np",
            padding="max_length",
            truncation=True,
            max_length=1024
        )
        
        input_ids = encoded["input_ids"].squeeze(0).tolist()
        batch_encoded.append(input_ids)
        labels.append(input_ids)

    # Pad both input_ids and labels
    padded_batch = tokenizer.pad({"input_ids": batch_encoded, "labels": labels}, padding=True, return_tensors="np")
    return padded_batch

# Tokenize the training and test datasets
columns_to_remove = ["endpoint", "question", "answer", "inputs", "output"]

tokenized_dataset_train = dataset['train'].map(
    collate_and_tokenize,
    batched=True,
    batch_size=1,
    remove_columns=columns_to_remove
)

tokenized_dataset_test = dataset['test'].map(
    collate_and_tokenize,
    batched=True,
    batch_size=1,
    remove_columns=columns_to_remove
)


os.environ["TOKENIZERS_PARALLELISM"] = "false"

torch.cuda.empty_cache()

print(torch.cuda.is_available())  # Should output: True
print(torch.cuda.get_device_name())



bnb_config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    llm_int8_enable_fp32_cpu_offload=False
)

from accelerate import init_empty_weights
from transformers import AutoModelForCausalLM

# Use the auto device map for proper placement
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="cuda:0",  # Automatically maps layers to devices
#device_map={'':torch.xpu.current_device()},
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    trust_remote_code=False,
    low_cpu_mem_usage=True 
)





def print_trainable_parameters(model):
    trainable_params=0
    all_params=0
    for _, param in model.named_parameters():
        all_params+=param.numel()
        if param.requires_grad:
            trainable_params+=param.numel()
    print(f"trainable params: {trainable_params} || all params: {all_params} || trainable%: {100 * trainable_params/all_params:.2f}")

print_trainable_parameters(model)

model.get_memory_footprint()

model.config.quantization_config


#gradient checkpointing to save memory
model.gradient_checkpointing_enable()
model.get_memory_footprint()

prepared_model=prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=True
)
prepared_model.get_memory_footprint()


peft_config=LoraConfig(
    r=256,
    lora_alpha=512,
    target_modules=[
        'q_proj',
        'k_proj',
        'v_proj',
        'o_proj',
        'gate_proj',
        'up_proj',
        'down_proj'
    ],
    bias="none",
    lora_dropout=0.05,
#    init_lora_weights=False,
    task_type=TaskType.CAUSAL_LM,
    modules_to_save= None
)
#prepared_model.add_adapter(peft_config, adapter_name="adapter_2")
lora_model=get_peft_model(prepared_model, peft_config)
lora_model.get_memory_footprint()

def print_trainable_parameters(model):
    trainable_params=0
    all_params=0
    for _, param in model.named_parameters():
        all_params+=param.numel()
        if param.requires_grad:
            trainable_params+=param.numel()
    print(f"trainable params: {trainable_params} || all params: {all_params} || trainable%: {100 * trainable_params/all_params:.2f}")

print_trainable_parameters(lora_model)


gc.collect()
torch.cuda.empty_cache()



training_args=TrainingArguments(
    output_dir='./ollamawork/Finetuning_2025/Finetune_colama-phi4/Result_phi4_colama_3',
    overwrite_output_dir=True,
    per_device_train_batch_size=3,
    per_device_eval_batch_size=3,
    gradient_accumulation_steps=5,
    gradient_checkpointing=True,  # Enable gradient checkpointing
    gradient_checkpointing_kwargs={"use_reentrant": False},
    warmup_steps=100,
    max_steps=500, # Total number of training steps
    #num_train_epochs=2, # Number of training epochs
    learning_rate=5e-04, # Learning rate
    weight_decay=0.01, # Weight decay
    optim="adamw_8bit", # Keep the optimizer state and quantize it
     bf16=True, # Do not supported in Kaggle environment, require Ampere....
#    fp16=True, # use fp16 16bit(mixed) precision training instead of 32-bit training.
    logging_dir='./logs',
    logging_strategy="steps",
    logging_steps=10,
    save_strategy="steps",
    save_steps=10,
    save_total_limit=4, # Limit the total number of checkpoints
    eval_strategy="steps",
    eval_steps=10,
    load_best_model_at_end=True, # Load the best model at the end of training,
    max_grad_norm=1.0,
    lr_scheduler_type= 'linear'
)

#lora_model.config.use_cache=False
lora_model.config_dropout=0.1
lora_model.config.use_cache=False

trainer=Trainer(
    model=lora_model,
    train_dataset=tokenized_dataset_train,
    eval_dataset=tokenized_dataset_test,
    args=training_args,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

start_time=time.time()
trainer.train(resume_from_checkpoint="./ollamawork/Finetuning_2025/Finetune_colama-phi4/Result_phi4_colama_3/checkpoint-100")
#trainer.train()
end_time=time.time()

training_time=end_time-start_time

print(f"Training completed in {training_time} seconds.")
trainer.save_model("./ollamawork/Finetuning_2025/Finetune_colama-phi4/Result_phi4_colama_3")
tokenizer.save_pretrained("./ollamawork/Finetuning_2025/Finetune_colama-phi4/Result_phi4_colama_3")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import pandas as pd
from accelerate import init_empty_weights
from transformers import BitsAndBytesConfig
import os

# Load the dataset
val = pd.read_excel("./ollamawork/Finetuning_2025/Finetune_colama-llama3.1/train_750_library_files.xlsx",sheet_name='eval')
#val = pd.read_excel(val)
# Define the model name
model_name = './ollamawork/Finetuning_2025/Finetune_colama-phi4/Result_phi4_colama_3'

# Load the tokenizer and model with 4-bit precision
tokenizer = AutoTokenizer.from_pretrained(model_name)


# Load the fine-tuned model
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16, device_map="auto")

# Set the padding token to eos_token
tokenizer.pad_token = tokenizer.eos_token
tokenizer.truncation_side = "left"

import re

def get_between_triple_quotes(string):
    triple_quote = '"""'
    start = string.find(triple_quote)  # Find the first triple quote
    end = string.find(triple_quote, start + 3)  # Find the next triple quote after the first
    if start != -1 and end != -1 and end > start:
        return string[start + 3:end].strip()  # Extract the content between the triple quotes
    else:
        return ""  # Return an empty string if triple quotes are not found

    
    
def filt_s(s):
    if s:  # Check if the string is not empty
        substrings = s.split('\n')  # Split the string by newlines
        if len(substrings) > 1:  # Check if there is more than one line
            desired_substring = substrings[1]  # Get the second line
            cleaned_substring = desired_substring.strip()  # Strip leading/trailing whitespace
            return cleaned_substring
        else:
            return s.strip()  # Return the string itself if it's a single line
    else:
        return '' 
def gen(endpoint,question,desc, inputs, output):
    new_prompt = f"""
<|im_start|>system<|im_sep|> 
Write down a python function that tests the API scenario described below. Make sure you use appropriate docstring while writing code. Below are the endpoint, inputs, and output of the function as well.<|im_end|>
<|im_start|>user<|im_sep|>
Endpoint:
{endpoint}\n
Description:
{question}\n
{desc}\n
Inputs:
{inputs}\n
Output:
{output}\n
Answer:\n
    """
    # Tokenize the input
    inputs = tokenizer(
        new_prompt,
        return_tensors="pt",
        return_attention_mask=True,  # Enable attention mask
        padding=True,
        truncation=True
    )

    # Move inputs to GPU
    inputs = {key: value.to('cuda') for key, value in inputs.items()}
    torch.manual_seed(42)

    # Generate text
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,  # Limit length to prevent excessive output
        eos_token_id=tokenizer.eos_token_id,  # Set end-of-sequence token
        repetition_penalty=1,  # Discourage repetitive generation
        temperature=0.1,
        do_sample=True
    )
    generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    return generated_text



n=0
#i=0
output_column_name = 'Generated Code'
if output_column_name not in val.columns:
    val[output_column_name] = "" 

for i in range(len(val) - n):
    # Generate code based on the test case summary and steps
    desc = filt_s(get_between_triple_quotes(val['Test_Code'][i+n]))
    generated_code = gen( val['Endpoint'][i+n], val['Description'][i+n], desc , val['Inputs'][i+n], val['Output'][i+n])
    
    # Store the extracted generated code in the DataFrame
    val.at[i + n, output_column_name] = generated_code
    
    # Save the updated DataFrame after each iteration to ensure progress is stored
    output_file = "./ollamawork/Finetuning_2025/Finetune_colama-phi4/Result_phi4_colama_0.1_3.xlsx"
    val.to_excel(output_file, index=False)

    # Print progress
    if (int(i+1+n))%100 == 0:
      print(f"Generated code for test case {i + 1 + n} is done. Saved to {output_file}.")
    

print(f"Generated code saved in column '{output_column_name}' and saved to {output_file}")

In [ ]:
import pandas as pd

# Load the Excel file
input_file = "./ollamawork/Finetuning_2025/Finetune_colama-phi4/Result_phi4_colama_0.1_3.xlsx"
df = pd.read_excel(input_file)

# Define the column containing the generated code
output_column_name = 'Generated Code'
output_txt_file = "generated_code_0.2_3.txt"

# Check if the column exists
if output_column_name in df.columns:
    with open(output_txt_file, "w", encoding="utf-8") as f:
        for index, code in enumerate(df[output_column_name].dropna(), start=1):
            f.write(f"### Test Case {index}\n")
            f.write(code + "\n\n\n\n\n")
            f.write('---------------------------------------------------------------------------------------------------')
    print(f"Generated code saved to {output_txt_file}")
else:
    print(f"Column '{output_column_name}' not found in the Excel file.")

Generated code saved to generated_code_0.2_3.txt


In [ ]:
import time
import gc
import os
import pandas as pd

# Assuming necessary libraries are imported for file operations and string handling
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load the Excel sheet
val = pd.read_excel("./ollamawork/Finetuning_2025/Finetune_colama-llama3.1/train_750_library_files.xlsx",sheet_name='eval')

# Helper functions
def get_between_triple_quotes(string):
    triple_quote = '"""'
    start = string.find(triple_quote)  # Find the first triple quote
    end = string.find(triple_quote, start + 3)  # Find the next triple quote after the first
    if start != -1 and end != -1 and end > start:
        return string[start + 3:end].strip()  # Extract the content between the triple quotes
    else:
        return ""  # Return an empty string if triple quotes are not found

    
    
def filt_s(s):
    if s:  # Check if the string is not empty
        substrings = s.split('\n')  # Split the string by newlines
        if len(substrings) > 1:  # Check if there is more than one line
            desired_substring = substrings[1]  # Get the second line
            cleaned_substring = desired_substring.strip()  # Strip leading/trailing whitespace
            return cleaned_substring
        else:
            return s.strip()  # Return the string itself if it's a single line
    else:
        return ''  # Return an empty string if the input is empty

# Write the formatted output to a file
n = 0
output_file_path = './ollamawork/Finetuning_2025/Finetune_colama-phi4/Ollama_val_original.txt'

with open(output_file_path, 'w') as file:
    for i in range(len(val) - n): 
        # Add spacing between entries
        file.write("\n\n")

        # Extract and format the fields
        endpoint = val.Endpoint[i + n]
        description = val.Description[i + n]
        inputs = val.Inputs[i + n] if type(val.Inputs[i + n]) == str else str('nan')
        output = val.Output[i + n] if type(val.Output[i + n]) == str else str('nan')
        answer = val.Test_Code[i + n]
        
        # Process the 'Answer' field
        processed_answer = filt_s(get_between_triple_quotes(answer)) if type(answer) == str else ""

        # Format the content
        content = f"""

systemWrite down a python function that tests the API scenario described below. Make sure you use appropriate docstring while writing code. Below are the endpoint, inputs, and output of the function as well.userEndpoint:
{endpoint}

Description:
{description}

{processed_answer}

Inputs:
{inputs}

Output:
{output}

Answer:
{answer}
"""
        # Write the formatted content to the file
        file.write(content)

        # Add a separator line
        file.write("\n\n\n\n\n")
        file.write('---------------------------------------------------------------------------------------------------\n')
        
        # Print progress
        print(f"{i + 1 + n} is done.")

print(f"Formatted output written to {output_file_path}.")